# Neural Forge - MCP Battle Tank Lab

In [1]:
# import asyncio
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerSse
import os
from IPython.display import Markdown, display
from datetime import datetime

print("Imports loaded")


Imports loaded


In [2]:
load_dotenv(override=True)

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is not set")

In [4]:
from IPython.display import display_markdown
from agents.mcp.server import MCPServerStreamableHttp



BATTLE_FIELD_URL = "https://battle-tank-arena.vercel.app/api/mcp"
TANK_NAME = "Hidden Tides"

print("URL: ", BATTLE_FIELD_URL)
print("NAME: ", TANK_NAME)


URL:  https://battle-tank-arena.vercel.app/api/mcp
NAME:  Hidden Tides


In [11]:
mcp_params = {
    "url": BATTLE_FIELD_URL,
    "headers": {
        "x-player-token": TANK_NAME,
    },
}

async with MCPServerStreamableHttp(mcp_params, client_session_timeout_seconds=30) as mcp_server:
    print("MCP server started")
    mcp_tools = await mcp_server.list_tools()


print("MCP tools:", mcp_tools)

print(len(mcp_tools), "tools", [t.name for t in mcp_tools])


MCP server started
MCP tools: [Tool(name='register', title=None, description='Register a new tank to join the game. Call this FIRST before any other action. Your tank name IS your token - reconnect to this MCP endpoint with header "x-player-token: YourTankName" to authenticate all future tool calls. If you lose connection, just register again with the same name to reconnect. Name must be unique, max 20 characters. Registration is only open while the game is in the lobby (waiting for players).', inputSchema={'type': 'object', 'properties': {'name': {'type': 'string', 'minLength': 1, 'maxLength': 20, 'description': 'Your tank name - must be unique, this becomes your token'}}, 'required': ['name'], 'additionalProperties': False, '$schema': 'http://json-schema.org/draft-07/schema#'}, outputSchema=None, icons=None, annotations=None, meta=None, execution=ToolExecution(taskSupport='forbidden')), Tool(name='get_valid_actions', title=None, description='Get ALL valid actions for your current tur

In [14]:
MODEL_NAME = "gpt-4.1-mini"

BATTLE_INSTRUCTIONS = """
You are Hidden Tides, an aggressive and intelligent battle tank agent competing in Battle Tank Arena.

## CORE RULE — NEVER IDLE
You operate in a continuous loop. When it is not your turn, you poll get_game_state 
repeatedly until it is. You NEVER output phrases like "I will wait", "I'll check back", 
or "it is not my turn yet" and stop. Saying you will wait IS NOT waiting — only 
calling get_game_state again is waiting. Never stop the loop until game status is 'ended'.

## IDENTITY & AUTH
- Your tank name AND authentication token is provided in the user message.
- On first run: call `register` with your exact name ONCE, only if the game is in lobby phase.
- To reconnect after disconnect: call `register` again with the same name.

## TURN STRUCTURE (60 seconds — act fast)
Each turn you have 2 dice. You MUST use both dice or your turn is skipped.
- `rotate` — FREE action, no die consumed. Use it to aim before shooting or moving.
- `move(die=N)` — consumes die N, moves you in your current facing direction.
- `fire(die=N)` — consumes die N, fires a shot from your current facing direction.

## EVERY TURN WORKFLOW
1. Call `get_game_state` → confirm it's your turn (currentTurnId matches your name).
2. Call `get_valid_actions` → get your position, dice values, all enemy positions, validShots, validMoves.
3. Decide your FULL plan for both dice before acting.
4. Execute actions. Never skip a die.

## TACTICAL DECISION PRIORITY (in order)
1. **KILL SHOT FIRST**: If `validShots` lists an enemy name (guaranteed hit) for any direction, rotate to that direction and fire immediately with whichever die is most effective. A confirmed kill is always the top priority.
2. **DOUBLE HIT**: If both dice offer confirmed hits on enemies, fire both — use each die separately after rotating to the right direction.
3. **SETUP SHOT + MOVE**: If only one die offers a confirmed hit, fire it. Then use the second die to reposition: move toward cover, away from enemies, or into a position where next turn's shot is likely to hit.
4. **REPOSITION + MISS SHOT**: If no confirmed hits are available, use one die to move to a better firing position. Use the second die to fire anyway — a near-miss pressures enemies and you MUST consume the die.
5. **SURVIVAL MOVE**: If an enemy has a clear shot on you next turn, prioritize moving out of their line of fire over offensive play.

## SHOT TARGETING RULES
- ONLY fire in directions listed in `validShots` from the latest `get_valid_actions` call.
- `validShots` entries with a non-null target name = guaranteed hit. Always chase these.
- `validShots` entries with null target = miss, but still fires the die (required if no better option).
- Never fire in a direction NOT in validShots — the shot will be blocked and the die is wasted.

## MOVEMENT RULES
- ONLY move in directions listed in `validMoves` from latest `get_valid_actions`.
- Tanks can pass through each other — only the final destination must be empty.
- Prefer positions that increase your shooting angles on enemies next turn.
- Avoid positions where multiple enemies can hit you simultaneously.

## ROTATION RULES
- Rotate is FREE — do it as many times as needed to face your chosen direction.
- Only directions in `validRotations` are allowed.
- Always rotate BEFORE firing or moving if you need to change direction.


## WINNING MINDSET
- Be aggressive. Every turn without a hit is a lost opportunity.
- Never idle — always move + shoot or shoot + reposition.
- Track enemy positions from `get_valid_actions` and anticipate their next moves.
- Low-score enemies are near elimination — prioritize finishing them off.
- If you're low on score, play defensively: move first, then fire.
"""

# USER_PROMPT = f"""You are tank '{TANK_NAME}'. Your name is also your auth token.
    
#     Step 1: Call get_game_state.   
#     - If status is 'lobby' and you are not yet registered, call register with your exact name.
#     - If status is 'running' and currentTurnId matches your name, it is your turn — take it now.
#     - If it is not your turn, call get_game_state again after a short wait until it is.
#     Step 2: On your turn, call get_valid_actions, form your full plan, then execute both dice. 
    
#     Do not skip a die. You have 60 seconds — move fast.
#     Play to win. Prioritize confirmed hits. Keep repositioning. Stay alive.
# """

USER_PROMPT = (
    f"You are tank '{TANK_NAME}'. Your name is also your auth token.\n\n"
    "STARTUP:\n"
    "  Call get_game_state. If status is 'lobby' and you are not registered, call register once.\n\n"
    "MAIN LOOP — repeat until the game ends:\n"
    "  1. Call get_game_state.\n"
    "  2. If status is 'ended', stop.\n"
    "  3. If currentTurnId is NOT your name, call get_game_state again immediately. "
    "     Do NOT say you are waiting. Do NOT pause. Just call get_game_state again.\n"
    "  4. If currentTurnId IS your name: call get_valid_actions, plan both dice, execute. "
    "     Use both dice. You have 60 seconds.\n"
    "  5. After your turn, go back to step 1.\n\n"
    "CRITICAL: Never stop polling. Never output 'I will wait'. "
    "The only way to wait for your turn is to keep calling get_game_state."
)

In [15]:


async with MCPServerStreamableHttp(
    mcp_params,
    client_session_timeout_seconds=600,
) as mcp_server:
    agent = Agent(
        name="MrDeeBattleTankBot",
        instructions=BATTLE_INSTRUCTIONS.strip(),
        mcp_servers=[mcp_server],
    )
    prompt = USER_PROMPT
    with trace("mrdee-battle-tank-lab"):
        result = await Runner.run(agent, prompt, max_turns=200)
    print(result.final_output)

The game state status is "ended." No further actions will be taken. If a new game starts in the future, please prompt again to participate as Hidden Tides.
